# 3. Load fundamental ratios for the ML value factor (Refinitiv, point-in-time)

The ML value factor scores stocks from **18 fundamental ratios** (`RATIO_COLUMNS` – P/E, P/B, EV/EBIT, RoE,
Debt/Equity, ...). This notebook pulls the raw accounting items from Refinitiv for a universe, derives the ratios
with one shared formula set (`data_engineering.fundamentals.ratios.compute_ratios`) and stores them in
`dbo.security_fundamentals`, one row per (security, ratio, fiscal period end).

Why store history rather than a snapshot? Because a backtest must only use what was **known at the time**.
`StaticFundamentalsProvider` serves the latest row per security with `effective_date <= as_of_date`, so a
2021 rebalance can never see a 2024 filing. `effective_date` is the fiscal **period end**; statements are published
weeks later, which is why the scoring side adds an `availability_lag_days` (notebook 04).

| Step | Library call |
|---|---|
| Resolve the universe | `data_engineering.loaders.portfolios.portfolio_member_ids` |
| Pull quarterly ratio history and upsert it, retrying missed names individually | `data_engineering.loaders.load_fundamentals_with_retry` |
| Coverage of what landed | `data_engineering.loaders.fundamentals_coverage` |
| Prove there is no look-ahead | `data_engineering.loaders.check_point_in_time` |

Re-running for the same window is safe: writes are a non-destructive MERGE.

## 3.1 Parameters

In [3]:
UNIVERSE = "SP500"          # portfolio whose members to load
FROM_DATE = "2020-01-01"    # first fiscal period end to keep
TO_DATE = "2026-08-31"      # history end / snapshot date
FREQUENCY = "Q"             # "Q" quarterly history (recommended) | "FY" single latest annual snapshot
WRITE_DB = True             # False: pull a sample and inspect only
SAMPLE_LIMIT = None         # e.g. 20 for a smoke test; None = whole universe
SOURCE_VENDOR = "refinitiv"

## 3.2 Connections and universe

In [4]:
import sys
import os

import pandas as pd

# Get the current notebook's directory and go up to parent
current_dir = os.getcwd()
parent_dir = os.path.dirname(os.path.dirname(current_dir))

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

print(f"Current directory: {current_dir}")
print(f"Added to path: {parent_dir}")
data_eng_path = os.path.join(parent_dir, 'data_engineering')
files_path = os.path.join(parent_dir, 'toolkit\\files')
print(f"Data engineering folder exists: {os.path.exists(data_eng_path)}")
print(f"Files folder exists: {os.path.exists(files_path)}")

Current directory: c:\SourceCode\InvestmentManagement\toolkit\notebooks
Added to path: c:\SourceCode\InvestmentManagement
Data engineering folder exists: True
Files folder exists: True


In [5]:
import logging

import pandas as pd

from data_engineering.database import database
from data_engineering.fundamentals import RATIO_COLUMNS, RefinitivFundamentalsProvider, StaticFundamentalsProvider
from data_engineering.loaders import check_point_in_time, fundamentals_coverage, load_fundamentals_with_retry
from data_engineering.loaders.portfolios import portfolio_member_ids
from data_engineering.refinitiv import ensure_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
ensure_session()
engine, connection, conn_str, session = database.get_db_connection()

members = portfolio_member_ids(engine, UNIVERSE)
master = database.read_security_master(session, engine)
universe = master[master["security_id"].isin(members)][["security_id", "name"]].rename(columns={"name": "symbol"}).drop_duplicates("security_id")
if SAMPLE_LIMIT:
    universe = universe.head(SAMPLE_LIMIT)
print(f"{len(universe)} securities in universe '{UNIVERSE}'")
universe.head()

2026-09-25 19:03:20,750 INFO httpx: HTTP Request: GET http://localhost:9005/api/status "HTTP/1.1 200 OK"
2026-09-25 19:03:20,766 INFO httpx: HTTP Request: POST http://localhost:9005/api/handshake "HTTP/1.1 200 OK"
2026-09-25 19:03:20,767 INFO data_engineering.refinitiv.session: LSEG session opened


Database connection successful.
636 securities in universe 'SP500'


,security_id,symbol
0,1,A.N
1,2,AAL.OQ
2,3,AAP.N
3,4,AAPL.OQ
4,5,ABBV.N


## 3.3 Inspect a small sample before writing

The provider resolves each security's exchange-qualified RIC (from `security_vendor_xref`, then LSEG
`symbol_conversion`), pulls the raw items per fiscal quarter and computes the ratios. `snapshot` is the latest
quarter per security; `history` has one row per quarter.

In [6]:
sample = universe.head(5)
provider = RefinitivFundamentalsProvider(orm_session=session, orm_engine=engine, frequency=FREQUENCY, start_date=FROM_DATE)
snapshot, history = provider.get_panel_history(sample, TO_DATE, frequency=FREQUENCY, start_date=FROM_DATE)
print("snapshot:", snapshot.shape, "| history rows:", None if history is None else len(history))
if history is not None:
    print("fiscal period ends:", history["effective_date"].min().date(), "->", history["effective_date"].max().date())
snapshot.round(3)

2026-09-25 19:03:32,841 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:03:45,417 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:04:02,995 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:04:03,072 INFO data_engineering.refinitiv.client: get_history chunk 1/1 OK: 53 rows x 75 cols


snapshot: (5, 18) | history rows: 165
fiscal period ends: 2019-09-28 -> 2026-06-30


,EV/EBIT,Op. In./(NWC+FA),P/E,P/B,P/S,Op. In./Interest Expense,Working Capital Ratio,RoE,ROCE,Debt/Equity,Debt Ratio,Cash Ratio,Asset Turnover,Gross Profit Margin,(CA-CL)/TA,RE/TA,EBIT/TA,Book Equity/TL
security_id,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3.4 Load the full universe

`load_fundamentals_with_retry` runs the bulk pull, checks which securities have no rows in the window and retries
those **one at a time** (a single-RIC request rarely hits the gateway timeout that makes a whole chunk disappear).

In [7]:
if WRITE_DB:
    summary = load_fundamentals_with_retry(universe, session, engine, FROM_DATE, TO_DATE, provider_name="refinitiv", frequency=FREQUENCY, source_vendor=SOURCE_VENDOR)
    print(f"rows written: {summary['rows_written']:,} | covered: {len(summary['covered'])} | still missing: {len(summary['missing'])}")
    if summary["missing"]:
        print("missing security_ids (typically delisted / renamed):", summary["missing"][:30])
else:
    print("WRITE_DB is False - nothing written.")

2026-09-25 19:04:05,347 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:04:05,550 INFO data_engineering.refinitiv.client: get_history chunk 1/43 OK: 88 rows x 225 cols
2026-09-25 19:04:08,867 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:04:09,053 INFO data_engineering.refinitiv.client: get_history chunk 2/43 OK: 33 rows x 225 cols
2026-09-25 19:04:17,962 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:04:32,684 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:04:50,333 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:05:08,027 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:05:25,711 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:05:25,922 INFO data_engineering.refinitiv.cli

Upserting 54992 security_fundamentals rows (non-destructive MERGE)...


2026-09-25 19:15:46,119 INFO data_engineering.fundamentals.store: wrote 54992 fundamentals rows from 'refinitiv'


Security fundamentals data successfully written (54992 rows upserted).


2026-09-25 19:15:49,407 INFO data_engineering.loaders.fundamentals_loader: coverage after initial pull: 603/636 securities
2026-09-25 19:15:52,592 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:15:52,595 WARNING data_engineering.refinitiv.client: get_history chunk 1/1 attempt 1/3 failed: LDError: Unable to resolve all requested identifiers in ['ABMD.OQ']. -- retry in 5s
2026-09-25 19:16:00,353 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:16:00,355 WARNING data_engineering.refinitiv.client: get_history chunk 1/1 attempt 2/3 failed: LDError: Unable to resolve all requested identifiers in ['ABMD.OQ']. -- retry in 5s
2026-09-25 19:16:07,892 INFO httpx: HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-09-25 19:16:07,893 ERROR data_engineering.refinitiv.client: get_history chunk 1/1 FAILED after 3 attempts: Unable to resolve all requested identifiers in ['ABMD.OQ'].
2026-09-25

rows written: 54,992 | covered: 603 | still missing: 33
missing security_ids (typically delisted / renamed): [6, 20, 22, 32, 55, 105, 149, 153, 164, 171, 172, 179, 210, 238, 243, 304, 326, 343, 409, 410, 419, 449, 481, 501, 505, 511, 546, 563, 581, 606]


## 3.5 What landed

In [8]:
rows = fundamentals_coverage(session, engine, universe["security_id"].tolist(), FROM_DATE, TO_DATE, SOURCE_VENDOR)
print(f"{len(rows):,} ratio rows | {rows['security_id'].nunique()} securities | {rows['effective_date'].nunique()} distinct period ends")
per_metric = rows.groupby("metric_type")["security_id"].nunique().reindex(RATIO_COLUMNS).rename("securities_with_data")
display(per_metric.to_frame().T)
demo_metrics = ["P/E", "P/B", "EV/EBIT", "RoE"]
covered_all = rows[rows["metric_type"].isin(demo_metrics)].groupby("security_id")["metric_type"].nunique()
sec0 = covered_all[covered_all == len(demo_metrics)].index[0] if (covered_all == len(demo_metrics)).any() else universe["security_id"].iloc[0]
demo = rows[rows["security_id"] == sec0].pivot_table(index="effective_date", columns="metric_type", values="metric_value")
demo[[c for c in demo_metrics if c in demo.columns]].round(3).tail()

48,730 ratio rows | 603 securities | 233 distinct period ends


metric_type,EV/EBIT,Op. In./(NWC+FA),P/E,P/B,P/S,Op. In./Interest Expense,Working Capital Ratio,RoE,ROCE,Debt/Equity,Debt Ratio,Cash Ratio,Asset Turnover,Gross Profit Margin,(CA-CL)/TA,RE/TA,EBIT/TA,Book Equity/TL
securities_with_data,485.0,NaN,486.0,486.0,468.0,540.0,508.0,603.0,603.0,603.0,603.0,NaN,584.0,518.0,508.0,599.0,603.0,603.0


metric_type,P/E,P/B,EV/EBIT,RoE
effective_date,,,,
2021-12-31,-5.835,-1.584,-9.059,0.272
2022-12-31,NaN,NaN,NaN,-0.022
2023-12-31,NaN,NaN,NaN,-0.158
2024-12-31,13.548,-2.882,12.876,-0.213
2025-12-31,91.193,-2.716,24.021,-0.030


## 3.6 Point-in-time guarantee

For every quarter end in the window, ask the database provider for the panel "as of" that date and confirm that
no returned ratio has an `effective_date` later than the date asked for. The result must be **0** leaked values.

In [9]:
static = StaticFundamentalsProvider(session, engine, source_vendor=SOURCE_VENDOR)
all_rows = database.read_security_fundamentals(session, engine, metric_type=None)
all_rows = all_rows[(all_rows["source_vendor"] == SOURCE_VENDOR) & all_rows["security_id"].isin(universe["security_id"])]
worst = check_point_in_time(static, universe, FROM_DATE, TO_DATE, all_rows)
print("maximum number of future-dated ratios visible on any quarter end (must be 0):", worst)

maximum number of future-dated ratios visible on any quarter end (must be 0): 0


In [10]:
session.close()
connection.close()

**Next:** notebook **04** trains the ML value models on this history and backtests a momentum + ML-value long/short
book. The same load can be run from the command line:

```
python toolkit/scripts/load_refinitiv_fundamentals.py --universe SP500 --frequency Q --start 2020-01-01
```